> 在线讲义：[第 25 章 · MLA：把 K、V 压成一个小向量存，用时再展开；RoPE 为什么要单独拆出来](https://wikieden.github.io/beyond-transformer/lessons/25-mla.html) · 本书《超越 Transformer》

# 第 25 章 · MLA：把 K、V 压成一个小向量存，用时再展开；RoPE 为什么要单独拆出来

配套讲义：[在线版](https://wikieden.github.io/beyond-transformer/lessons/25-mla.html) · 仓库里 `beyond-transformer/notes/lessons/25-mla.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 吸收 | W_UKᵀq·c | `scores(hq, hs, M)` | 两种写法差 2.2e-16 |
| 第三步 RoPE 冲突 | W_UKᵀR_{j−t}ᵀq | `absorbed_query_with_rope(hq, M, rel)` | 随相对位置变 |
| 第四步 cache | (d_c + d_h^R)·l | `cache_elems(kind)` | 34,560 vs 1,966,080 |

推理时 W_UK 可以吸收进 query（q·(W_UK c) = (W_UKᵀ q)·c），不用真的展开；可是 RoPE 夹在中间就吸收不了，
所以单独拆出一小段带 RoPE 的 key（decoupled RoPE）。cache 账：每个 token 每层 d_c + d_h^R 个数。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
D, DC, DH = 16, 4, 8          # 隐层宽、压缩维数、每头维数（单头演示）

**函数 `mats`**：小工具。

In [ ]:
def mats(seed=0):
    rng = np.random.default_rng(seed)
    return {
        "W_DKV": rng.normal(0, 0.3, (DC, D)),    # 压缩：h → c
        "W_UK": rng.normal(0, 0.3, (DH, DC)),    # 展开成 key
        "W_UV": rng.normal(0, 0.3, (DH, DC)),    # 展开成 value
        "W_Q": rng.normal(0, 0.3, (DH, D)),
    }

**函数 `scores`**：一个 query（隐状态 hq）对若干个历史隐状态 hs 的分数：先展开再点积 vs 把 W_UK 吸收进 query。

In [ ]:
def scores(hq, hs, M):
    """一个 query（隐状态 hq）对若干个历史隐状态 hs 的分数：先展开再点积 vs 把 W_UK 吸收进 query。"""
    q = M["W_Q"] @ hq
    c = np.array([M["W_DKV"] @ h for h in hs])                 # cache 里存的是 c（DC 维）
    expand = np.array([q @ (M["W_UK"] @ ci) for ci in c])      # 先展开 k = W_UK c
    q_abs = M["W_UK"].T @ q                                    # 吸收：q′ = W_UKᵀ q（只算一次）
    absorbed = c @ q_abs
    return expand, absorbed

**函数 `rot_matrix`**：小工具。

In [ ]:
def rot_matrix(pos, theta=0.5, d=DH):
    R = np.eye(d)
    for i in range(0, d, 2):
        a = pos * theta / (1 + i)
        c, s = np.cos(a), np.sin(a)
        R[i : i + 2, i : i + 2] = [[c, -s], [s, c]]
    return R

**函数 `absorbed_query_with_rope`**：q 在位置 m、k 在位置 n：(R_m q)ᵀ R_n W_UK c = qᵀ R_{n−m} W_UK c。要吸收就得算 q′ = W_UKᵀ R_{n−m}ᵀ q，

In [ ]:
def absorbed_query_with_rope(hq, M, rel):
    """q 在位置 m、k 在位置 n：(R_m q)ᵀ R_n W_UK c = qᵀ R_{n−m} W_UK c。要吸收就得算 q′ = W_UKᵀ R_{n−m}ᵀ q，
    它依赖相对位置 rel = n − m：每个 key 位置都得重新算一遍（等于把 key 展开），吸收就没意义了。"""
    q = M["W_Q"] @ hq
    return M["W_UK"].T @ rot_matrix(rel).T @ q

**函数 `cache_elems`**：每个 token 的 KV cache 元素个数（DeepSeek-V2 表 1）。

In [ ]:
def cache_elems(kind, n_h=128, d_h=128, layers=60, n_g=8, d_c=512, d_rope=64):
    """每个 token 的 KV cache 元素个数（DeepSeek-V2 表 1）。"""
    per = {"MHA": 2 * n_h * d_h, "GQA": 2 * n_g * d_h, "MQA": 2 * d_h, "MLA": d_c + d_rope}[kind]
    return per * layers

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
M = mats()
rng = np.random.default_rng(1)
hs = rng.normal(size=(5, D))
print("第一步 cache 里只存压缩向量 c")
print(f"  隐层 {D} 维 → c {DC} 维；展开出的 k、v 各 {DH} 维（真实设定：5120 → 512，128 个头 × 128 维）")
print("\n第二步 W_UK 吸收进 query：不用真的展开")
e, a = scores(hs[-1], hs, M)
print(f"  先展开再点积：{np.round(e, 4).tolist()}")
print(f"  吸收进 query：{np.round(a, 4).tolist()}（最大差 {np.abs(e - a).max():.1e}）")

In [ ]:
print("\n第三步 RoPE 夹在中间就吸收不了")
for rel in (0, -1, -4):
    print(f"  相对位置 {rel:2d}：要用的 q′ = W_UKᵀ R({rel})ᵀ q = {np.round(absorbed_query_with_rope(hs[-1], M, rel), 4).tolist()}")
print("  q′ 随相对位置变：每个 key 都要一个不同的 q′，没法只算一次；DeepSeek-V2 另拆一小段 d_h^R = 64 维的 key 专门带 RoPE")

In [ ]:
print("\n第四步 cache 账（60 层，每个 token 的元素数）")
for k in ("MHA", "GQA", "MQA", "MLA"):
    print(f"  {k}：{cache_elems(k):,}")
print(f"  MLA 每层 {512 + 64} = 4.5 × d_h，相当于只有 2.25 组的 GQA；比 MHA 小 {cache_elems('MHA') / cache_elems('MLA'):.1f} 倍")

## 3. 代码里才看得到的

- **训练时一般不吸收**：训练是整段并行算的，直接展开 K、V 更简单；吸收只在逐 token 推理、需要省读写时才用。
- **这里只演示单头**：真实的 W_UK 是每个头一块（128 个头 × 128 维），吸收后每个头得到一个 512 维的 q′；所有头共用同一份 c。

## 4. 自己改一改

- 把 `DC` 从 4 改成 8（和 k 一样大），吸收还成立吗？cache 还省吗？
- 给 `cache_elems` 换成 DeepSeek-V3 的设定（61 层、d_c = 512、d_h^R = 64），每个 token 多少个元素？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 存压缩包，用时再解压：K、V 共用一个五百一十二维的 c。
> q 点 W 乘 c，等于 W 转置乘 q 再点 c：展开矩阵吸收进 query，不用解压。
> RoPE 夹在中间随位置变，吸收就失效：单独拆一段六十四维、各头共享的 k^R 带位置。
> 每层五百七十六个数，相当于两组多的 GQA，比 MHA 小五十七倍。